In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

C:\Users\borja\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.11.0' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import (
C:\Users\borja\AppData\Roaming\Python\Python313\site-packages\pandas\core\arrays\masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
c:\Users\borja\anaconda3\Lib\site-packages\seaborn\_statistics.py:32: UserWarning: A NumPy version >=1.23.5 and <2.5.0 is required for this version of SciPy (detected version 2.5.4)
  from scipy.stats import gaussian_kde


In [2]:
results_23 = pd.read_csv('../data/results_23.csv')
races_23 = pd.read_csv('../data/races_23.csv')
constructors_23 = pd.read_csv('../data/constructors_23.csv')
lap_times_23 = pd.read_csv('../data/lap_times_23.csv')

In [3]:
print(f'Results: {len(results_23)} | Races: {len(races_23)} | Constructors: {len(constructors_23)}')
print(f'Lap times: {len(lap_times_23):,} filas')

Results: 440 | Races: 22 | Constructors: 10
Lap times: 24,386 filas


In [ ]:
raceids_2023 = set(races_23['raceId'])
raceids_lap = set(lap_times_23['raceId'])
cobertura = raceids_2023.issubset(raceids_lap)
print(f'Cobertura 2023: {"COMPLETA" if cobertura else "FALTAN DATOS"}')

Cobertura 2023: ✅ COMPLETA


In [5]:
# Tabla de puntos F1 2023
PUNTOS_2023 = {
    1: 25, 2: 18, 3: 15, 4: 12, 5: 10,
    6: 8, 7: 6, 8: 4, 9: 2, 10: 1
}

def calcular_puntos(posicion):
    '''Puntos según tabla 2023. Posiciones >10 = 0'''
    return PUNTOS_2023.get(int(posicion), 0)

# Vectorizado
calcular_puntos_vec = np.vectorize(calcular_puntos)

# Mostrar tabla
print('Tabla F1 2023:')
for pos, pts in PUNTOS_2023.items():
    print(f'  P{pos}: {pts} pts')

Tabla F1 2023:
  P1: 25 pts
  P2: 18 pts
  P3: 15 pts
  P4: 12 pts
  P5: 10 pts
  P6: 8 pts
  P7: 6 pts
  P8: 4 pts
  P9: 2 pts
  P10: 1 pts


In [7]:
abandonos_23 = results_23[results_23['statusGeneric'].isin(['Accident', 'Mechanical issue'])].copy()

print(f'Total abandonos con causa en 2023: {len(abandonos_23)}')
print('\nDistribución por tipo:')
print(abandonos_23['statusGeneric'].value_counts())

print(f'\nAbandonos con raceId: {abandonos_23["raceId"].notna().sum()}')
print(f'Abandonos con driverId: {abandonos_23["driverId"].notna().sum()}')
print(f'Abandonos con grid: {abandonos_23["grid"].notna().sum()}')

Total abandonos con causa en 2023: 58

Distribución por tipo:
statusGeneric
Accident            33
Mechanical issue    25
Name: count, dtype: int64

Abandonos con raceId: 58
Abandonos con driverId: 58
Abandonos con grid: 58


In [8]:
# 1. Obtener última vuelta por (raceId, driverId) en lap_times
ultima_vuelta = lap_times_23.groupby(['raceId', 'driverId'])['lap'].max().reset_index()

# 2. Merge para obtener posición en esa última vuelta
ultima_vuelta = ultima_vuelta.merge(
    lap_times_23[['raceId', 'driverId', 'lap', 'position']],
    on=['raceId', 'driverId', 'lap'],
    how='left'
)
ultima_vuelta = ultima_vuelta.rename(columns={'position': 'ultima_posicion'})

# 3. Merge con abandonos
abandonos_con_pos = abandonos_23.merge(
    ultima_vuelta[['raceId', 'driverId', 'ultima_posicion']],
    on=['raceId', 'driverId'],
    how='left'
)

# 4. Fallback: si no hay datos en lap_times, usar grid
abandonos_con_pos['posicion_contrafactual'] = abandonos_con_pos['ultima_posicion'].fillna(abandonos_con_pos['grid'])

In [9]:
# 5. Calcular puntos perdidos
abandonos_con_pos['puntos_perdidos'] = calcular_puntos_vec(abandonos_con_pos['posicion_contrafactual'])

# 6. Estadísticas
con_cobertura = abandonos_con_pos['ultima_posicion'].notna().sum()
sin_cobertura = abandonos_con_pos['ultima_posicion'].isna().sum()

print(f'Abandonos con datos en lap_times: {con_cobertura}')
print(f'Abandonos usando fallback (grid): {sin_cobertura}')
print(f'Cobertura: {con_cobertura/len(abandonos_con_pos)*100:.1f}%')

Abandonos con datos en lap_times: 49
Abandonos usando fallback (grid): 9
Cobertura: 84.5%


In [10]:
# Agrupar por statusGeneric
puntos_por_tipo = abandonos_con_pos.groupby('statusGeneric').agg({
    'puntos_perdidos': ['sum', 'count', 'mean']
}).round(2)

puntos_por_tipo.columns = ['Puntos perdidos', 'Número abandonos', 'Media pts/abandono']
puntos_por_tipo = puntos_por_tipo.reset_index()

# Mostrar resultados
print('=== PUNTOS PERDIDOS POR TIPO ===')
print(puntos_por_tipo.to_string(index=False))

=== PUNTOS PERDIDOS POR TIPO ===
   statusGeneric  Puntos perdidos  Número abandonos  Media pts/abandono
        Accident               65                33                1.97
Mechanical issue               28                25                1.12


In [11]:
puntos_accident = puntos_por_tipo[puntos_por_tipo['statusGeneric'] == 'Accident']['Puntos perdidos'].values[0]
puntos_mecanico = puntos_por_tipo[puntos_por_tipo['statusGeneric'] == 'Mechanical issue']['Puntos perdidos'].values[0]
n_accident = puntos_por_tipo[puntos_por_tipo['statusGeneric'] == 'Accident']['Número abandonos'].values[0]
n_mecanico = puntos_por_tipo[puntos_por_tipo['statusGeneric'] == 'Mechanical issue']['Número abandonos'].values[0]

print(f'\nAccidentes: {puntos_accident} puntos perdidos ({n_accident} abandonos)')
print(f'Mecánicos: {puntos_mecanico} puntos perdidos ({n_mecanico} abandonos)')
print(f'Diferencia: {puntos_accident - puntos_mecanico} puntos')
print(f'Ratio (Acc/Mec): {puntos_accident / puntos_mecanico:.2f}x')


Accidentes: 65 puntos perdidos (33 abandonos)
Mecánicos: 28 puntos perdidos (25 abandonos)
Diferencia: 37 puntos
Ratio (Acc/Mec): 2.32x


In [13]:
# CRITERIO 1: Accident > Mechanical issue
criterio_1 = puntos_accident > puntos_mecanico

# CRITERIO 2: Ratio >= 1.5x
criterio_2 = (puntos_accident / puntos_mecanico) >= 1.5

print('=== VERIFICACIÓN DE CRITERIOS (A PRIORI) ===')
print(f'Criterio 1 (Accident > Mechanical): {"CUMPLE" if criterio_1 else "NO CUMPLE"}')
print(f'  {puntos_accident} > {puntos_mecanico}: {criterio_1}')
print()
print(f'Criterio 2 (Ratio >= 1.5x): {"CUMPLE" if criterio_2 else "NO CUMPLE"}')
print(f'  {puntos_accident / puntos_mecanico:.2f}x >= 1.5x: {criterio_2}')
print()
print(f'- H4 SE CONFIRMA' if (criterio_1 and criterio_2) else '- H4 NO SE CONFIRMA')

=== VERIFICACIÓN DE CRITERIOS (A PRIORI) ===
Criterio 1 (Accident > Mechanical): CUMPLE
  65 > 28: True

Criterio 2 (Ratio >= 1.5x): CUMPLE
  2.32x >= 1.5x: True

- H4 SE CONFIRMA


In [14]:
# 1. Calcular puntos reales en carreras terminadas
resultados_terminados = results_23[results_23['statusGeneric'] == 'Finished'].copy()
resultados_terminados['puntos_reales'] = calcular_puntos_vec(resultados_terminados['position'])

# 2. Media de puntos por piloto
media_puntos_piloto = resultados_terminados.groupby('driverId')['puntos_reales'].mean()
conteos_piloto = resultados_terminados.groupby('driverId').size()

# 3. Para pilotos con <3 carreras terminadas, usar media del equipo
driver_constructor_map = results_23.drop_duplicates(['driverId', 'constructorId'])[['driverId', 'constructorId']]
media_puntos_equipo = resultados_terminados.groupby('constructorId')['puntos_reales'].mean()

# Crear serie final de media
media_piloto_final = media_puntos_piloto.copy()
for driver_id in media_puntos_piloto.index:
    if conteos_piloto[driver_id] < 3:
        constructor_id = driver_constructor_map[driver_constructor_map['driverId'] == driver_id]['constructorId'].values
        if len(constructor_id) > 0:
            media_piloto_final[driver_id] = media_puntos_equipo[constructor_id[0]]

# 4. Aplicar a abandonos
abandonos_con_pos['puntos_contrafactual_B'] = abandonos_con_pos['driverId'].map(media_piloto_final)

# 5. Agrupar por tipo
puntos_B_por_tipo = abandonos_con_pos.groupby('statusGeneric').agg({
    'puntos_contrafactual_B': ['sum', 'count', 'mean']
}).round(2)
puntos_B_por_tipo.columns = ['Puntos esperados', 'Número abandonos', 'Media']
puntos_B_por_tipo = puntos_B_por_tipo.reset_index()

# Extraer valores
puntos_B_accident = puntos_B_por_tipo[puntos_B_por_tipo['statusGeneric'] == 'Accident']['Puntos esperados'].values[0]
puntos_B_mecanico = puntos_B_por_tipo[puntos_B_por_tipo['statusGeneric'] == 'Mechanical issue']['Puntos esperados'].values[0]

# Verificar criterios
criterio_B_1 = puntos_B_accident > puntos_B_mecanico
criterio_B_2 = (puntos_B_accident / puntos_B_mecanico) >= 1.5

print('=== CONTRAFACTUAL B (Sensibilidad) ===')
print(puntos_B_por_tipo.to_string(index=False))
print(f'\nAccidentes: {puntos_B_accident:.1f} pts esperados')
print(f'Mecánicos: {puntos_B_mecanico:.1f} pts esperados')
print(f'Ratio: {puntos_B_accident / puntos_B_mecanico:.2f}x')
print(f'\nB1 (Acc > Mec): {"✅" if criterio_B_1 else "❌"}')
print(f'B2 (Ratio >= 1.5x): {"✅" if criterio_B_2 else "❌"}')

=== CONTRAFACTUAL B (Sensibilidad) ===
   statusGeneric  Puntos esperados  Número abandonos  Media
        Accident            133.40                33   4.04
Mechanical issue             93.88                25   3.76

Accidentes: 133.4 pts esperados
Mecánicos: 93.9 pts esperados
Ratio: 1.42x

B1 (Acc > Mec): ✅
B2 (Ratio >= 1.5x): ❌


In [15]:
# Conteo simple: número de abandonos por tipo
conteo_accident = len(abandonos_23[abandonos_23['statusGeneric'] == 'Accident'])
conteo_mecanico = len(abandonos_23[abandonos_23['statusGeneric'] == 'Mechanical issue'])

# Criterios para conteo simple
criterio_b_1 = conteo_accident > conteo_mecanico
criterio_b_2 = (conteo_accident / conteo_mecanico) >= 1.5

print('=== SENSIBILIDAD (b): CONTEO SIMPLE ===')
print(f'Accidentes: {conteo_accident} abandonos')
print(f'Mecánicos: {conteo_mecanico} abandonos')
print(f'Ratio: {conteo_accident / conteo_mecanico:.2f}x')
print()
print(f'b1 (Acc > Mec): {"✅" if criterio_b_1 else "❌"}')
print(f'b2 (Ratio >= 1.5x): {"✅" if criterio_b_2 else "❌"}')

=== SENSIBILIDAD (b): CONTEO SIMPLE ===
Accidentes: 33 abandonos
Mecánicos: 25 abandonos
Ratio: 1.32x

b1 (Acc > Mec): ✅
b2 (Ratio >= 1.5x): ❌
